# Partie 5 — Modèle de gravité

Le modèle de gravité prédit le **montant attendu d'un sinistre, sachant qu'il y a eu un sinistre** : $\mathbb{E}[S \mid S>0]$. Il est appris uniquement sur les **2 917 contrats sinistrés**.

Cette partie a été développée par mon binôme. Nous gardons les **trois gravités qu'il a retenues**, chacune ayant un score Kaggle (combinée à la régression logistique de fréquence) :

| Gravité | Principe | Kaggle |
|---|---|---|
| 1. Gamma de référence | régression Gamma (lien log) sur 3 variables : âge du véhicule, type de contrat, usage | 1365.943 |
| 2. Gamma + correction XGBoost | la Gamma est corrigée par un petit XGBoost très régularisé (50 % Gamma, 50 % Gamma corrigée) | 1365.913 |
| 3. Gamma + correction par courbes | on part de la gravité 2 et on ajoute 25 % d'une correction Ridge avec courbes (splines) et catégories sur les variables véhicule, conducteur et contrat | 1365.745 |

**Comment ces gravités sont conservées.** Pour garder le projet simple, le code complet du binôme n'est plus dans le dossier (il reste dans l'historique git). Ses prédictions ont été extraites une fois, le 20 octobre 2026, avec son propre code, dans `Data/predictions/` :
- `binome_composantes_test.csv` : modèles entraînés sur les 50 000 contrats, prédictions sur le test Kaggle ; elles reproduisent ses soumissions Kaggle à moins de $10^{-8}$ € près ;
- `binome_composantes_oof_train.csv` : prédictions **hors pli** sur le train (5 plis par client, colonne `pli`) : chaque contrat est prédit par des modèles qui ne l'ont pas vu.

La **gravité 1** est réécrite ci-dessous en quelques lignes, et on vérifie qu'elle redonne exactement les prédictions extraites. Les gravités 2 et 3 sont **figées** : on les utilise telles quelles.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Racine du projet : le dossier qui contient Data/ (lancement depuis notebooks/ ou depuis la racine)
RACINE = next(p for p in [Path.cwd(), Path.cwd().parent] if (p / "Data" / "train.csv").exists())
PREDICTIONS = RACINE / "Data" / "predictions"

train = pd.read_csv(RACINE / "Data" / "train.csv", dtype={"code_postal": str})
test = pd.read_csv(RACINE / "Data" / "test.csv", dtype={"code_postal": str})
oof = pd.read_csv(PREDICTIONS / "binome_composantes_oof_train.csv")   # prédictions hors pli sur le train
binome_test = pd.read_csv(PREDICTIONS / "binome_composantes_test.csv")  # prédictions sur le test Kaggle
assert (oof["index"].values == train["index"].values).all() and (binome_test["index"].values == test["index"].values).all()
print("train :", train.shape, "| test :", test.shape, "| plis :", sorted(oof["pli"].unique()))

train : (50000, 33) | test : (50000, 28) | plis : [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]


## 1. La Gamma de référence, réécrite

- **Variables** : `anciennete_vehicule` (valeur manquante → médiane, puis standardisation), `type_contrat` et `utilisation` en one-hot. Une modalité qui a moins de 20 sinistres (ou inconnue) va dans une colonne « rare ».
- **Modèle** : `GammaRegressor` de scikit-learn, lien logarithmique, régularisation `alpha = 0,1`. La loi Gamma convient à des montants **positifs et asymétriques** : l'erreur permise grandit avec le montant.
- **Apprentissage** : uniquement sur les contrats sinistrés.

In [2]:
from sklearn.linear_model import GammaRegressor

CAT_GRAVITE = ["type_contrat", "utilisation"]


def apprendre_gravite(sinistres, seuil_rare=20):
    # Paramètres appris sur les sinistrés d'apprentissage seulement
    p = {"mediane_age": sinistres["anciennete_vehicule"].median()}
    age = sinistres["anciennete_vehicule"].fillna(p["mediane_age"])
    p["moyenne_age"], p["ecart_age"] = age.mean(), age.std(ddof=0)
    for col in CAT_GRAVITE:
        effectifs = sinistres[col].value_counts()
        p[f"modalites_{col}"] = sorted(effectifs[effectifs >= seuil_rare].index)
    return p


def preparer_gravite(X, p):
    Z = pd.DataFrame(index=X.index)
    Z["anciennete_vehicule"] = (X["anciennete_vehicule"].fillna(p["mediane_age"]) - p["moyenne_age"]) / p["ecart_age"]
    for col in CAT_GRAVITE:
        for modalite in p[f"modalites_{col}"]:
            Z[f"{col}_{modalite}"] = (X[col] == modalite).astype(float)
        Z[f"{col}_rare"] = (~X[col].isin(p[f"modalites_{col}"])).astype(float)  # rares et inconnues
    return Z


def gamma_reference(apprentissage, X):
    # Apprend la Gamma sur les sinistrés de « apprentissage », prédit le montant pour X
    sinistres = apprentissage[apprentissage["nombre_sinistres"] == 1]
    p = apprendre_gravite(sinistres)
    modele = GammaRegressor(alpha=0.1, solver="lbfgs", max_iter=5000, tol=1e-5)
    modele.fit(preparer_gravite(sinistres, p), sinistres["montant_sinistre"])
    return modele.predict(preparer_gravite(X, p))


# Vérification 1 : appris sur tout le train, prédit sur le test
gravite_test = gamma_reference(train, test)
print(f"Test : écart max avec la gravité extraite = {np.abs(gravite_test - binome_test['gravite_gamma_reference']).max():.4f} €")

# Vérification 2 : hors pli, sur les mêmes 5 plis que l'extraction
gravite_oof = np.zeros(len(train))
for k in sorted(oof["pli"].unique()):
    dans_pli = (oof["pli"] == k).values
    gravite_oof[dans_pli] = gamma_reference(train[~dans_pli], train[dans_pli])
print(f"Hors pli : écart max avec la gravité extraite = {np.abs(gravite_oof - oof['gravite_gamma_reference']).max():.4f} €")

Test : écart max avec la gravité extraite = 0.0000 €


Hors pli : écart max avec la gravité extraite = 0.0000 €


→ La Gamma réécrite redonne **exactement** les prédictions extraites (écart maximal 0,0000 €), sur le test comme hors pli. La gravité 1 est donc entièrement reproductible depuis ce notebook.

## 2. Comparaison des trois gravités, hors pli

On compare les trois gravités **sur les contrats sinistrés uniquement** (la gravité n'a de sens que s'il y a un sinistre), avec leurs prédictions hors pli :
- **RMSE** et **MAE** du montant (en €) ;
- **ratio** somme prédite / somme observée (1 = pas de biais global) ;
- les mêmes mesures sur les **sinistres les plus coûteux** (10 % les plus chers), pour la question du TD : le modèle est-il fiable sur les gros sinistres ?

In [3]:
GRAVITES = {"1. Gamma de référence": "gravite_gamma_reference",
            "2. Gamma + XGBoost": "gravite_gamma_xgboost",
            "3. Gamma + courbes": "gravite_gamma_courbes"}

sinistres_oof = oof[oof["nombre_sinistres"] == 1]
observe = sinistres_oof["montant_sinistre"]
seuil_couteux = observe.quantile(0.9)
couteux = observe >= seuil_couteux

lignes = {"Montant moyen (constante)": np.full(len(observe), observe.mean())}
lignes.update({nom: sinistres_oof[col] for nom, col in GRAVITES.items()})
comparaison = pd.DataFrame({
    nom: {"RMSE (€)": np.sqrt(np.mean((prevu - observe) ** 2)),
          "MAE (€)": np.mean(np.abs(prevu - observe)),
          "ratio prédit / observé": np.sum(prevu) / observe.sum(),
          "ratio sur les 10 % plus coûteux": np.sum(np.asarray(prevu)[couteux.values]) / observe[couteux].sum()}
    for nom, prevu in lignes.items()}).T
print(f"Sinistrés : {len(observe)} | seuil des 10 % les plus coûteux : {seuil_couteux:.0f} €")
comparaison.round(3)

Sinistrés : 2917 | seuil des 10 % les plus coûteux : 3337 €


,RMSE (€),MAE (€),ratio prédit / observé,ratio sur les 10 % plus coûteux
Montant moyen (constante),1463.704,953.495,1.000,0.349
1. Gamma de référence,1452.520,931.271,0.998,0.360
2. Gamma + XGBoost,1451.285,931.154,1.001,0.363
3. Gamma + courbes,1452.679,930.267,1.000,0.364


→ **Les trois gravités font à peine mieux qu'un montant constant** : RMSE de 1 451 à 1 453 € contre 1 464 €, et MAE de 930 à 931 € contre 953 €, soit moins de 1 % de gain en RMSE. Le montant d'un sinistre est surtout aléatoire : les variables disponibles en expliquent très peu.
- **Pas de biais global** : la somme prédite est égale à la somme observée (ratio entre 0,998 et 1,001).
- **Pas fiables sur les sinistres coûteux** : pour les 10 % les plus chers (au-dessus de 3 337 €), les modèles ne prédisent qu'environ 36 % du montant observé (ratio de 0,360 à 0,364), à peine mieux que la constante (0,349). Le modèle prédit un montant moyen ; il ne sait pas repérer à l'avance un gros sinistre.
- Selon la mesure, ce n'est pas la même gravité qui gagne (RMSE : gravité 2, MAE : gravité 3) : les écarts entre elles sont très petits. Le critère qui compte pour Kaggle est la **prime**, comparée dans le notebook 03.